# Further Feature Combination Evaluation (SMOTE dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_PATH = "/content/drive/My Drive/WVS_happiness_study"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
X_train = pd.read_csv(f"{DRIVE_DATA_PATH}/data/train_sets/X_train_final_smote.csv")
X_test = pd.read_csv(f"{DRIVE_DATA_PATH}/data/test_sets/X_test_final.csv")
X_val = pd.read_csv(f"{DRIVE_DATA_PATH}/data/val_sets/X_val_final.csv")

y_train = pd.read_csv(f"{DRIVE_DATA_PATH}/data/train_sets/y_train_final_smote.csv")
y_test = pd.read_csv(f"{DRIVE_DATA_PATH}/data/test_sets/y_test_final.csv")
y_val = pd.read_csv(f"{DRIVE_DATA_PATH}/data/val_sets/y_val_final.csv")

## 5 Combinations of Features

In [ ]:
feature_sets = {

    "set_1": [
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3",
        "trust3",
        "ethical80",
        "civic53",
        "compassion1",
        "compassion2",
        "compassion3",
        "compassion4",
        "growth16",
        "age",
        "marital_status",
        "religious1",
        "income_scale"
    ],

    "set_2": [
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3",
        "trust3",
        "ethical80",
        "civic53",
        "compassion2",
        "compassion3",
        "growth16",
        "age",
        "marital_status"
    ],

    "set_3": [ # maslow needs only
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3"
    ],

    "set_4": [ # maturity of heart only
        "trust3",
        "ethical80",
        "civic53",
        "compassion1",
        "compassion2",
        "growth16"
    ],

    "set_5": [ # hybrid
        "sact",
        "physio1",
        "safety5",
        "trust3",
        "ethical80"
    ],

    "set_6": [ # top 3
        "sact",
        "physio1",
        "safety5"
    ]
}

Create data splits.

In [4]:
data_splits = {}

for name, features in feature_sets.items():
    data_splits[name] = {
        "X_train": X_train[features],
        "X_val": X_val[features],
        "X_test": X_test[features],
        "feature_columns": features
    }

In [5]:
import warnings
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
pd.pandas.set_option("display.max_columns", None)

In [6]:
models = [
    {
        "name": "RandomForest",
        "estimator": RandomForestClassifier(random_state=42)
    },
    {
        "name": "LightGBM",
        "estimator": LGBMClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [-1, 5, 8]
        }
    },
    {
        "name": "LogisticRegression",
        "estimator": LogisticRegression(random_state=42, max_iter=3000)
    },
    {
        "name": "NaiveBayes",
        "estimator": GaussianNB()
    }
]


## Model Training

In [7]:
import sys
sys.path.append(f"{DRIVE_DATA_PATH}/code/my_ml_tools")
import training

all_results = {}
all_models = {}

for combo_name, data in data_splits.items():
    print(f"\n==============================")
    print(f"Running feature set: {combo_name}")
    print(f"==============================")

    results_df, best_models =  training.train_ensemble_models(
        data["X_train"], y_train,
        data["X_val"], y_val,
        data["X_test"], y_test,
        models,
        sort_by= "Test Accuracy"
    )

    all_results[combo_name] = results_df
    all_models[combo_name] = best_models


Running feature set: set_1
y_train: Converted to 1D pd automatically...
y_test: Converted to 1D pd automatically...
y_val: Converted to 1D pd automatically...

Training RandomForest...

Training LightGBM...
[LightGBM] [Info] Number of positive: 56869, number of negative: 56869
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006691 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 113738, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [8]:
from sklearn.metrics import classification_report

OUTPUT_PATH = f"{DRIVE_DATA_PATH}/data/feature_sets_performance/model_feature_comparison_smote.txt"

with open(OUTPUT_PATH, "w") as f:

    for combo_name, df in all_results.items():

        print(f"Evaluating: {combo_name}")

        f.write("\n" + "="*70 + "\n")
        f.write(f"FEATURE SET: {combo_name}, Predictors: {len(feature_sets[combo_name])}\n")
        f.write("="*70 + "\n\n")

        # Metrics table
        f.write(df.to_string(index=False))
        f.write("\n\n")
        f.write("Classification Reports:\n")

        # 🔥 IMPORTANT: use EXACT training feature columns
        X_test_subset = data_splits[combo_name]["X_test"]

        for model_name, model_info in all_models[combo_name].items():

            model = model_info["model"]

            # 🔥 HARD FIX: align columns EXACTLY to training time
            try:
                # safest direct prediction
                y_pred = model.predict(X_test_subset)

            except Exception:
                # fallback: force alignment to model's expected features (if available)
                if hasattr(model, "feature_names_in_"):
                    X_aligned = X_test_subset.reindex(
                        columns=model.feature_names_in_,
                        fill_value=0
                    )
                    y_pred = model.predict(X_aligned)
                else:
                    raise

            f.write(f"\n\n--- {model_name} ---\n")
            f.write(classification_report(y_test, y_pred, digits=4))

        f.write("\n\n")

print("INFO: Evaluation completed successfully!")

Evaluating: set_1
Evaluating: set_2
Evaluating: set_3
Evaluating: set_4
Evaluating: set_5
Evaluating: set_6
INFO: Evaluation completed successfully!


Show the weighted avg for precision, recall, f1-score with test accuracy.

In [9]:
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report


def _flatten_target_values(y):
    if isinstance(y, (pd.Series, pd.DataFrame)):
        return np.ravel(y.to_numpy())
    return np.ravel(y)


def _predict_with_training_columns(model, X_test_subset):
    try:
        return model.predict(X_test_subset)
    except Exception:
        if hasattr(model, "feature_names_in_"):
            X_aligned = X_test_subset.reindex(
                columns=model.feature_names_in_,
                fill_value=0
            )
            return model.predict(X_aligned)
        raise


def show_weighted_average_metrics_table(
    all_models,
    data_splits,
    y_test,
    all_results=None,
    display_by_feature_set=True
):
    y_true = _flatten_target_values(y_test)
    summary_rows = []

    for combo_name, model_infos in all_models.items():
        X_test_subset = data_splits[combo_name]["X_test"]

        if all_results is not None and combo_name in all_results:
            result_df = all_results[combo_name]
            if "Model" in result_df.columns:
                model_names = result_df["Model"].tolist()
            else:
                model_names = list(model_infos.keys())
        else:
            model_names = list(model_infos.keys())

        for model_name in model_names:
            if model_name not in model_infos:
                continue

            model_info = model_infos[model_name]
            model = model_info["model"] if isinstance(model_info, dict) and "model" in model_info else model_info
            y_pred = _predict_with_training_columns(model, X_test_subset)
            report = classification_report(
                y_true,
                y_pred,
                output_dict=True,
                zero_division=0
            )
            weighted_avg = report["weighted avg"]

            summary_rows.append({
                "feature_set": combo_name,
                "model": model_name,
                "weighted_precision": weighted_avg["precision"],
                "weighted_recall": weighted_avg["recall"],
                "weighted_f1": weighted_avg["f1-score"],
                "test_accuracy": accuracy_score(y_true, y_pred)
            })

    metrics_df = pd.DataFrame(summary_rows)

    if display_by_feature_set:
        for combo_name, combo_df in metrics_df.groupby("feature_set", sort=False):
            print(f"\nFEATURE SET: {combo_name}")
            display(combo_df.drop(columns="feature_set").reset_index(drop=True).round(4))
    else:
        display(metrics_df.round(4))

    return metrics_df


weighted_average_metrics_df = show_weighted_average_metrics_table(
    all_models=all_models,
    data_splits=data_splits,
    y_test=y_test,
    all_results=all_results,
    display_by_feature_set=True
)

weighted_average_metrics_df


FEATURE SET: set_1


,model,weighted_precision,weighted_recall,weighted_f1,test_accuracy
0,LightGBM,0.8578,0.8744,0.8612,0.8744
1,RandomForest,0.8562,0.8742,0.8587,0.8742
2,LogisticRegression,0.8614,0.7681,0.7974,0.7681
3,NaiveBayes,0.8554,0.7670,0.7957,0.7670



FEATURE SET: set_2


,model,weighted_precision,weighted_recall,weighted_f1,test_accuracy
0,LightGBM,0.8581,0.8747,0.8615,0.8747
1,RandomForest,0.8448,0.8633,0.8503,0.8633
2,LogisticRegression,0.8618,0.7685,0.7977,0.7685
3,NaiveBayes,0.8540,0.7650,0.7940,0.7650



FEATURE SET: set_3


...


FEATURE SET: set_5


,model,weighted_precision,weighted_recall,weighted_f1,test_accuracy
0,RandomForest,0.8527,0.8715,0.8561,0.8715
1,LightGBM,0.8548,0.8695,0.8594,0.8695
2,LogisticRegression,0.8607,0.7716,0.7999,0.7716
3,NaiveBayes,0.8574,0.7617,0.7920,0.7617



FEATURE SET: set_6


,model,weighted_precision,weighted_recall,weighted_f1,test_accuracy
0,RandomForest,0.8523,0.8661,0.8572,0.8661
1,LightGBM,0.8544,0.8637,0.8583,0.8637
2,LogisticRegression,0.8596,0.7765,0.8034,0.7765
3,NaiveBayes,0.8596,0.7765,0.8034,0.7765


,feature_set,model,weighted_precision,weighted_recall,weighted_f1,test_accuracy
0,set_1,LightGBM,0.857750,0.874376,0.861194,0.874376
1,set_1,RandomForest,0.856157,0.874236,0.858700,0.874236
2,set_1,LogisticRegression,0.861386,0.768144,0.797411,0.768144
3,set_1,NaiveBayes,0.855351,0.767020,0.795678,0.767020
4,set_2,LightGBM,0.858112,0.874657,0.861504,0.874657
5,set_2,RandomForest,0.844804,0.863275,0.850331,0.863275
6,set_2,LogisticRegression,0.861815,0.768496,0.797736,0.768496
7,set_2,NaiveBayes,0.854002,0.764983,0.793951,0.764983
8,set_3,RandomForest,0.853265,0.871988,0.856452,0.871988
9,set_3,LightGBM,0.854670,0.863838,0.858502,0.863838


Export Logistic models for SMOTE setting

In [10]:
import joblib

logistic_models_per_combo = {}

for combo_name in all_results.keys():

    # search for LogisticRegression in trained models
    if "LogisticRegression" not in all_models[combo_name]:
        print(f"{combo_name}: LogisticRegression not found, skipping...")
        continue

    # get trained logistic regression model
    log_model = all_models[combo_name]["LogisticRegression"]["model"]

    # store in dict (optional)
    logistic_models_per_combo[combo_name] = log_model

    # save model
    filename = f"{DRIVE_DATA_PATH}/model/smote_logistic/smote_logistic_model_{combo_name}.pkl"
    joblib.dump(log_model, filename)

    print(f"{combo_name}: Saved LogisticRegression → {filename}")

set_1: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_1.pkl
set_2: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_2.pkl
set_3: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_3.pkl
set_4: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_4.pkl
set_5: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_5.pkl
set_6: Saved LogisticRegression → /content/drive/My Drive/WVS_happiness_study/model/smote_logistic/smote_logistic_model_set_6.pkl
